# SCAPE Tutorial

This notebook walks through the full SCAPE pipeline on simulated data:

1. **Simulate** paired pre/post data with confounding and collider features
2. **Baseline** — causal estimate on raw features (biased due to collider)
3. **Collider removal** — joint entropic OT couples post-period cells to pre-period counterparts
4. **Diagnostics** — propensity score recovery from OT coupling
5. **ATE estimation** — cross-fitted AIPW on OT-corrected embeddings
6. **ITE estimation** — individual treatment effects via OT coupling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.decomposition import PCA

import SCAPE

## 1. Simulate data

`simulate_observed_confounder` generates a paired pre/post experiment where:

- **U** — true (unobserved) confounders that affect both treatment assignment and outcome
- **C** — collider features: caused by both Y (outcome) and T (treatment) in the post period
- **K** — the observed feature matrix, which concatenates U and C

Key parameters:
- `tau` — true average treatment effect
- `alpha_y` — strength of the Y→C collider path
- `alpha_t` — strength of the T→C collider path (post period only)
- `U_simulator` — manifold structure of the confounders (`continuity_generator` for branching, `cluster_generator` for clusters)

In [ ]:
data = SCAPE.simulate_observed_confounder(
    n_slides_pre=3,
    n_slides_post=3,
    d_u=25,
    d_c=5,
    tau=0.5,
    U_simulator=SCAPE.continuity_generator,
    w_t_mu=0,
    w_t_sd=1,
    t_intercept=-0.1,
    alpha_y=2,
    alpha_t=4,
    add_batch=False,
    seed=42,
)

is_post    = data["is_post"] == 1
is_treated = data["T"] == 1

print(f"Total cells : {len(data['T'])}")
print(f"Pre-period  : {(~is_post).sum()}")
print(f"Post-period : {is_post.sum()}  "
      f"(treated={( is_post & is_treated).sum()}, "
      f"untreated={(is_post & ~is_treated).sum()})")
print(f"True tau    : 0.5")

### Helper: standardise features

In [ ]:
def center_scale(X, eps=1e-8):
    X = np.array(X, dtype=float).copy()
    std = X.std(axis=0)
    std[std < eps] = 1.0
    return (X - X.mean(axis=0)) / std

## 2. Baseline causal estimates

We run AIPW on two embeddings to establish baselines:

- **Ground truth (U)** — using the true unobserved confounders; this is the oracle upper bound
- **Raw (K)** — using the observed features; the collider C biases the estimate

In [ ]:
# Ground truth: true confounder U (oracle)
U         = center_scale(data["X"])
U_pca     = PCA().fit_transform(U)
res_truth = SCAPE.aipw_ate_crossfit(U_pca, data["T"], data["Y"].reshape(-1, 1))
print(f"ATE (oracle U)  : {res_truth['ate']:.3f}")

In [ ]:
# Raw observed features K — PCA fit on untreated cells only
K     = center_scale(data["K"])
pca_k = PCA()
pca_k.fit(K[data["T"] == 0])
K_pca   = pca_k.transform(K)
res_raw = SCAPE.aipw_ate_crossfit(K_pca, data["T"], data["Y"].reshape(-1, 1))
print(f"ATE (raw K)     : {res_raw['ate']:.3f}  ← biased by collider C")

In [ ]:
# Visualise the raw embedding coloured by pre/post and propensity score
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plt.sca(axes[0])
SCAPE.scatter(K_pca, group=data["is_post"], treatment=data["T"],
              group_title="pre / post")
plt.title("Raw K — pre vs post")
plt.sca(axes[1])
SCAPE.scatter(K_pca, group=data["p"], continuous=True, group_title="propensity")
plt.title("Raw K — true propensity score")
plt.tight_layout()
plt.show()

## 3. Collider removal via Joint Entropic OT

`fit_jot` solves a joint entropic OT problem that couples each post-period cell
to a distribution over pre-period cells.  Because the pre-period cells have not
yet been exposed to treatment, conditioning on this coupling removes the
collider path **T → C ← Y**.

Key `JointEntropicConfig` parameters:
- `epsilon` — entropic regularisation strength; smaller = sharper coupling
- `beta_treated` — relative cost weight for the treated sub-problem; controls how aggressively treated cells are matched

In [ ]:
cfg = SCAPE.JointEntropicConfig(
    epsilon=0.01,
    beta_treated=0.1,
    device="cpu",      # set to "cuda" if a GPU is available
    n_iters=5000,
    lr=0.5,
    verbose_every=500,
    tol_col=1e-6,
    tol_row=1e-6,
    stable_patience=20,
    lr_patience=80,
    lr_decay=0.5,
    min_lr=5e-4,
    stop_patience=300,
)

result = SCAPE.fit_jot(
    X=K_pca,
    is_post=data["is_post"],
    is_treated=data["T"],
    cfg=cfg,
)

## 4. Propensity score diagnostics

The column sums of `P_t` — the treated sub-coupling — reflect how much
treated-cell mass is transported to each pre-period cell.  This is a
proxy for the pre-period propensity score.  A high Spearman correlation
with the true propensity confirms the OT coupling has correctly identified
the confounder structure.

In [ ]:
residual_pre = result["P_t"].sum(axis=0)   # shape (n_pre,)
corr, pval   = spearmanr(data["p"][~is_post], residual_pre)
print(f"Spearman r (OT propensity vs true pre-period propensity): {corr:.3f}  (p={pval:.2e})")

plt.figure(figsize=(5, 4))
plt.scatter(data["p"][~is_post], residual_pre, s=4, alpha=0.4)
plt.xlabel("True propensity score (pre)")
plt.ylabel("P_t column sum")
plt.title(f"Propensity recovery  (r = {corr:.3f})")
plt.tight_layout()
plt.show()

## 5. ATE estimation on OT-corrected embeddings

For each post-period cell we sample a pre-period "match" according to
the OT coupling weights.  This gives a corrected embedding that no longer
contains the collider signal.  We repeat this 60 times and average the
resulting AIPW estimates.

In [ ]:
# Sparsify the full post→pre coupling (keeps 90 % of mass per row)
post_pre_map = SCAPE.row_mass_sparsify(result["P_full"], keep_mass=0.9)

# Draw 60 sampled corrected embeddings; shape: (60, n_post, d)
map_sample = SCAPE.sample_map_projection(
    post_pre_map, K_pca[~is_post], n_samples=60, random_state=42
)

ATE_list        = []
propensity_list = []
m0_list         = []
m1_list         = []

for X_corrected in map_sample:
    res = SCAPE.aipw_ate_crossfit(
        X_corrected,
        data["T"][is_post],
        data["Y"][is_post].reshape(-1, 1),
    )
    ATE_list.append(res["ate"])
    propensity_list.append(res["e_hat"])
    m0_list.append(res["m0_hat"])
    m1_list.append(res["m1_hat"])

ate_ot = np.mean(ATE_list)
print(f"ATE after collider removal: {ate_ot:.3f}  (true tau = 0.5)")

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(ATE_list, bins=15, edgecolor="white")
plt.axvline(0.5,        color="red",   lw=2, linestyle="--", label=f"true tau = 0.5")
plt.axvline(ate_ot,     color="black", lw=2, linestyle="-",  label=f"mean ATE = {ate_ot:.3f}")
plt.xlabel("ATE estimate")
plt.ylabel("Count")
plt.title("Distribution of AIPW estimates across 60 OT samples")
plt.legend()
plt.tight_layout()
plt.show()

## 6. ITE estimation

### 6a. ITE via OT coupling

Using the sparsified untreated (`G1`) and treated (`G2`) sub-couplings,
each pre-period cell gets a weighted average of treated and untreated
post-period outcomes.  Their difference is a direct ITE estimate.

In [ ]:
G1_map = SCAPE.row_mass_sparsify(result["P_u"].T, keep_mass=0.9)  # (n_pre, n_untreated_post)
G2_map = SCAPE.row_mass_sparsify(result["P_t"].T, keep_mass=0.9)  # (n_pre, n_treated_post)

ITE = (
    G2_map @ data["Y"][is_post & is_treated][:, None]
    - G1_map @ data["Y"][is_post & ~is_treated][:, None]
)

print(f"ITE mean (OT coupling): {ITE.mean():.3f}  (true tau = 0.5)")

### 6b. AIPW-OW ITE on aggregated nuisance estimates

Averaging the 60 nuisance model fits (outcome models and propensity)
reduces variance before computing the doubly robust AIPW-OW score.

In [ ]:
m0_mean   = np.mean(np.stack(m0_list),         axis=0)
m1_mean   = np.mean(np.stack(m1_list),         axis=0)
prop_mean = np.mean(np.stack(propensity_list), axis=0)

aipw_ave = SCAPE.aipw_ite_ow(
    data["Y"][is_post].reshape(-1, 1),
    data["T"][is_post],
    m0_mean, m1_mean, prop_mean,
)

print(f"AIPW-OW mean: {np.mean(aipw_ave):.3f}  (true tau = 0.5)")

## Summary

In [ ]:
rows = [
    ("True tau",                    0.5),
    ("ATE — oracle U",              res_truth["ate"]),
    ("ATE — raw K (biased)",        res_raw["ate"]),
    ("ATE — after collider removal", ate_ot),
    ("ITE mean (OT coupling)",      float(ITE.mean())),
    ("AIPW-OW mean",                float(np.mean(aipw_ave))),
]

print(f"{'Estimator':<35} {'Value':>8}")
print("-" * 44)
for label, val in rows:
    print(f"{label:<35} {val:>8.3f}")